In [ ]:
# CRAG , LangGraph

import os
from dotenv import load_dotenv
from typing import List
from typing_extensions import TypedDict
from langgraph.graph import END, StateGraph
from langchain_openai import ChatOpenAI
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()
llm = ChatOpenAI(model="gpt-5.4-nano", temperature=0)
embeddings = HuggingFaceEmbeddings(model_name="jhgan/ko-sroberta-multitask")
vector_db = Chroma(persist_directory="./chroma_db_session", embedding_function=embeddings)
retriever = vector_db.as_retriever(search_kwargs={"k": 2})

In [43]:
# 1. State 정의: 노드 간에 전달될 데이터 구조
class GraphState(TypedDict):
    question: str
    documents: List[str]
    generation: str
    web_search_needed: bool

# 2. Node 정의
def retrieve(state: GraphState):
    print("  -> [Node] Retrieve: 문서 검색 중...")
    question = state["question"]
    docs = retriever.invoke(question)    
    return {"documents": docs, "question": question}

def grade_documents(state: GraphState):
    print("  -> [Node] Grade: 검색된 문서의 관련성 자체 평가 중...")
    question = state["question"]
    docs = state["documents"]
    
    prompt = ChatPromptTemplate.from_template(
        "다음 문서가 사용자의 질문과 관련이 있는지 평가하세요.\n"
        "관련이 있으면 'yes', 없으면 'no'라고만 답하세요.\n\n"
        "문서: {context}\n질문: {question}"
    )
    chain = prompt | llm | StrOutputParser()
    
    filtered_docs = []
    web_search_needed = False
    
    for idx, doc in enumerate(docs):
        score = chain.invoke({"context": doc.page_content, "question": question}).strip().lower()
        if "yes" in score:
            print(f"     - 문서 {idx+1}: [PASS] 관련성 높음")
            filtered_docs.append(doc)
        else:
            print(f"     - 문서 {idx+1}: [FAIL] 관련성 낮음 (환각 위험)")
            web_search_needed = True
            
    if not filtered_docs:
        web_search_needed = True
        
    return {"documents": filtered_docs, "web_search_needed": web_search_needed}

def rewrite_query(state: GraphState):
    print("  -> [Node] Rewrite: 관련 문서 부족으로 쿼리 재작성(폴백) 수행 중...")
    question = state["question"]
    prompt = ChatPromptTemplate.from_template(
        "다음 질문을 검색 엔진이나 외부 웹 지식에서 더 잘 찾을 수 있도록 구체적으로 재작성하세요.\n"
        "원래 질문: {question}\n재작성된 질문:"
    )
    chain = prompt | llm | StrOutputParser()
    better_question = chain.invoke({"question": question})
    print(f"     - 원본 쿼리: {question}")
    print(f"     - 개선된 쿼리: {better_question}")
    return {"question": better_question}

def generate(state: GraphState):
    print("  -> [Node] Generate: 최종 응답 생성 중...")
    question = state["question"]
    docs = state["documents"]
    if docs:
        context = "\n\n".join([doc.page_content for doc in docs])   
    
        prompt = ChatPromptTemplate.from_template(
            "다음 문맥을 바탕으로 질문에 답하세요.\n\n문맥: {context}\n질문: {question}"
        )
        chain = prompt | llm | StrOutputParser()
        generation = chain.invoke({"context": context, "question": question})
    
        for doc in docs:
            filename = doc.metadata.get('filename','N/A')
            f_type = doc.metadata.get('file_type','N/A')
            cat = doc.metadata.get('category','N/A')
            generation += f'\n\n -출처:{filename}  파일 타입:{f_type}  카테고리:{cat}'
    else:
        generation = '관련 문서를 찾지못했습니다. (외부 지식으로 대처)'

    return {"generation": generation}

In [45]:
# 그래프컴파일 및 엣지 연결
def decide_to_generate(state: GraphState):
    if state['documents'] and state["web_search_needed"]:
        print("  -> [Condition] 일부 문서가 부적합하나 적합한 문서들이 있어 해당 문서를 기반으로 Generate 노드로 분기합니다.")
        return "generate"
    elif state["web_search_needed"]:
        print("  -> [Condition] 일부 문서가 부적합하여 Rewrite 노드로 분기합니다.")
        return "rewrite_query"
    else:
        print("  -> [Condition] 문서가 완벽히 관련성이 있어 Generate 노드로 분기합니다.")
        return "generate"

workflow = StateGraph(GraphState)
workflow.add_node("retrieve", retrieve)
workflow.add_node("grade_documents", grade_documents)
workflow.add_node("rewrite_query", rewrite_query)
workflow.add_node("generate", generate)

# 노드 흐름 연결
workflow.set_entry_point("retrieve")
workflow.add_edge("retrieve", "grade_documents")
workflow.add_conditional_edges(
    "grade_documents", 
    decide_to_generate, 
    {"rewrite_query": "rewrite_query", "generate": "generate"}
)
workflow.add_edge("rewrite_query", "generate")
workflow.add_edge("generate", END)

app = workflow.compile()

print("[테스트 1] 문서에 없는 일반적인 질문 (회사 규정)")
result1 = app.invoke({"question": "회사 환불 정책이 어떻게 되나요?"})
print(f"결과 1: {result1['generation']}\n")

print("\n\n[테스트 2] RAG에대해서 알려주세요 - 문서에 있음")
result2 = app.invoke({"question": "RAG에 대해서 알려주세요"})
print(f"결과 2: {result2['generation']}")

print("\n\n[테스트 3] 대회 규칙에 대해서 알려주세요 - 문서에 있음")
result3 = app.invoke({"question": "대회 규칙에 대해서 알려주세요"})
print(f"결과 3: {result3['generation']}")

[테스트 1] 문서에 없는 일반적인 질문 (회사 규정)
  -> [Node] Retrieve: 문서 검색 중...
  -> [Node] Grade: 검색된 문서의 관련성 자체 평가 중...
     - 문서 1: [FAIL] 관련성 낮음 (환각 위험)
     - 문서 2: [FAIL] 관련성 낮음 (환각 위험)
  -> [Condition] 일부 문서가 부적합하여 Rewrite 노드로 분기합니다.
  -> [Node] Rewrite: 관련 문서 부족으로 쿼리 재작성(폴백) 수행 중...
     - 원본 쿼리: 회사 환불 정책이 어떻게 되나요?
     - 개선된 쿼리: 다음처럼 더 구체적으로 재작성해 보세요(괄호 안만 원하시는 대로 채우면 됩니다).

1) **온라인 쇼핑몰 환불 정책 문의**
- “(쇼핑몰/회사명)에서 주문한 상품의 **환불 절차와 환불 기간**은 어떻게 되나요? 결제수단별(카드/계좌이체/간편결제)로 환불까지 걸리는 시간도 알려주세요.”

2) **구독/서비스(정기결제) 환불 정책 문의**
- “(서비스명) **구독 해지 시 환불 가능 여부**와 **환불 기준(사용 기간, 요금제, 프로모션 적용 여부)**은 어떻게 되나요?”

3) **티켓/예약 환불 정책 문의**
- “(예매 사이트/공연·행사명) 예매한 티켓의 **환불 가능 기간**, **취소 수수료**, **환불 방식(전액/부분환불)**은 어떻게 되나요? (출발/공연일 기준)”  

4) **오프라인 구매(매장) 환불 정책 문의**
- “(매장명/브랜드)에서 (상품명) 구매 후 **환불/교환 가능 기간**, **영수증/구매기록 필요 여부**, **수수료 또는 차감 조건**은 무엇인가요?”

5) **제품 하자/오배송 환불 정책 문의**
- “(회사명)에서 **불량품 또는 오배송**을 받았을 때 환불/교환 절차와 **교환·환불 소요 기간**, **반품 배송비 부담 주체**는 어떻게 되나요?”

원하시면 **회사명(또는 링크), 구매 형태(상품/구독/티켓), 결제수단, 주문일/희망 환불 